In [16]:
import os
import pandas as pd
import cv2

CSV_PATH = "/Users/artemsaltanovskyi/Documents/2025-2026-2/2MCTE2-Advancded_AI/exam/AIProject2026/Data/Csv/G.csv"
IMAGES_DIR = "/Users/artemsaltanovskyi/Documents/2025-2026-2/2MCTE2-Advancded_AI/exam/AIProject2026/Data/FramesFromCsv"
LABELS_DIR = "/Users/artemsaltanovskyi/Documents/2025-2026-2/2MCTE2-Advancded_AI/exam/AIProject2026/Data/LabelsFromCsv"

os.makedirs(LABELS_DIR, exist_ok=True)

CLASS_MAP = {
    "car": 0,
    "bus": 1,
    "truck": 2,
    "trunk": 2
}

df = pd.read_csv(CSV_PATH)
df["name"] = df["name"].astype(str).str.strip().str.lower()
df["name"] = df["name"].replace({"trunk": "truck"})
df = df[df["name"].isin(CLASS_MAP.keys())].copy()

image_files = sorted([
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

for image_name in image_files:
    frame_num = int(os.path.splitext(image_name)[0].split("_")[-1])
    image_path = os.path.join(IMAGES_DIR, image_name)
    label_path = os.path.join(LABELS_DIR, os.path.splitext(image_name)[0] + ".txt")

    img = cv2.imread(image_path)
    if img is None:
        continue

    h, w = img.shape[:2]
    rows = df[df["frame_num"] == frame_num]

    lines = []

    for _, row in rows.iterrows():
        xmin = max(0, min(float(row["xmin"]), w - 1))
        ymin = max(0, min(float(row["ymin"]), h - 1))
        xmax = max(0, min(float(row["xmax"]), w - 1))
        ymax = max(0, min(float(row["ymax"]), h - 1))

        if xmax <= xmin or ymax <= ymin:
            continue

        x_center = ((xmin + xmax) / 2) / w
        y_center = ((ymin + ymax) / 2) / h
        box_w = (xmax - xmin) / w
        box_h = (ymax - ymin) / h

        class_id = CLASS_MAP[row["name"]]
        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")

    with open(label_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

print("Done")

Done
